# Run benchmark test for NPE

In [1]:
import numpy as np
from scipy import stats
import torch
from tqdm.auto import tqdm
import warnings
from torch.distributions import Uniform
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.inference import NLE_A
from sbi.diagnostics import run_sbc, check_sbc
import sys
sys.path.append('../pysimARG')
from discrete_uniform import DiscreteUniform

torch_device = "cpu"

warnings.filterwarnings("ignore", category=UserWarning)

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load simulation data

Load genome data and clonal tree.

In [2]:
clonal_edge = np.loadtxt("../data/ClonalOrigin/clonal_edge.csv", delimiter=",", dtype=float)
clonal_node_height = np.loadtxt("../data/ClonalOrigin/clonal_node_height.csv", delimiter=",", dtype=float)

In [3]:
clonal_edge, clonal_node_height

(array([[3.10000000e+01, 1.40000000e+01, 1.80220314e-03],
        [3.10000000e+01, 2.90000000e+01, 1.80220314e-03],
        [3.20000000e+01, 2.80000000e+01, 5.97471972e-03],
        [3.20000000e+01, 2.60000000e+01, 5.97471972e-03],
        [3.30000000e+01, 3.00000000e+00, 8.25024952e-03],
        [3.30000000e+01, 2.70000000e+01, 8.25024952e-03],
        [3.40000000e+01, 2.20000000e+01, 1.30036692e-02],
        [3.40000000e+01, 1.00000000e+01, 1.30036692e-02],
        [3.50000000e+01, 1.10000000e+01, 2.34636374e-02],
        [3.50000000e+01, 1.80000000e+01, 2.34636374e-02],
        [3.60000000e+01, 3.30000000e+01, 1.80941931e-02],
        [3.60000000e+01, 2.10000000e+01, 2.63444427e-02],
        [3.70000000e+01, 1.70000000e+01, 3.56575696e-02],
        [3.70000000e+01, 6.00000000e+00, 3.56575696e-02],
        [3.80000000e+01, 1.20000000e+01, 3.73864931e-02],
        [3.80000000e+01, 3.20000000e+01, 3.14117734e-02],
        [3.90000000e+01, 3.00000000e+01, 3.85383268e-02],
        [3.900

In [4]:
theta_test = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta_sbc.csv', delimiter=",")
x_test = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x_sbc.csv', delimiter=",")

print(theta_test.shape, x_test.shape)
theta_test = torch.tensor(theta_test, device=torch_device)
theta_test = theta_test.to(torch.float32)
theta_test_numpy = theta_test.cpu().numpy()

x_test = torch.tensor(x_test, device=torch_device)
x_test = x_test.to(torch.float32)
x_test_numpy = x_test.cpu().numpy()

(1000, 3) (1000, 46)


In [5]:
nan_row = np.where(np.isnan(x_test_numpy))[0]
nan_row

array([865, 899])

In [6]:
theta_test = theta_test[~np.isnan(x_test_numpy).any(axis=1)]
x_test = x_test[~np.isnan(x_test_numpy).any(axis=1)]
theta_test.shape, x_test.shape

(torch.Size([998, 3]), torch.Size([998, 46]))

In [7]:
theta_test_numpy = theta_test.cpu().numpy()
x_test_numpy = x_test.cpu().numpy()
theta_test_numpy.shape, x_test_numpy.shape

((998, 3), (998, 46))

In [8]:
theta1 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta1.csv', delimiter=",")
x1 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x1.csv', delimiter=",")

theta2 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta2.csv', delimiter=",")
x2 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x2.csv', delimiter=",")

theta3 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta3.csv', delimiter=",")
x3 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x3.csv', delimiter=",")

theta4 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta4.csv', delimiter=",")
x4 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x4.csv', delimiter=",")

x = np.vstack([x1, x2, x3, x4])
theta = np.vstack([theta1, theta2, theta3, theta4])

print(theta.shape, x.shape)

(50000, 3) (50000, 46)


In [9]:
drop_indices = np.unique(np.concatenate([np.where(np.isnan(x))[0], np.where(np.isinf(x))[0]]))
drop_indices

array([  114,   681,   706,  1448,  2554,  2818,  7211,  7282,  7329,
        7392,  8938,  9827,  9973, 10223, 10788, 12192, 13567, 14388,
       14653, 20248, 21831, 22683, 23165, 24293, 24995, 26727, 27136,
       28568, 29040, 29263, 30213, 31152, 31611, 32633, 33029, 33132,
       34118, 34453, 34659, 35788, 35955, 40017, 40926, 41346, 42815,
       43524, 43687, 47022, 49215, 49429, 49517, 49777])

In [10]:
theta = np.delete(theta, drop_indices, axis=0)
x = np.delete(x, drop_indices, axis=0)

theta = torch.tensor(theta, device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = torch.tensor(x, device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

In [11]:
theta.shape, x.shape

(torch.Size([49948, 3]), torch.Size([49948, 46]))

## Benchmark setting

In [12]:
budgets = [200, 500, 1000, 2000, 5000, 10000, 49948]

In [13]:
def SBC_KStest(ranks, num_posterior_samples, parameter_labels):
    print("Kolmogorov-Smirnov Test Results (SBC):")
    print("-" * 40)

    num_dimensions = ranks.shape[1] 

    ks_results = []
    p_values = []
    for dim in range(num_dimensions):
        normalized_ranks = ranks[:, dim] / num_posterior_samples
        ks_stat, p_value = stats.kstest(normalized_ranks, 'uniform')
        ks_results.append(ks_stat)
        p_values.append(p_value)

        print(f"Parameter:        {parameter_labels[dim]}")
        print(f"KS Statistic (D): {ks_stat:.4f}")
        print(f"p-value:          {p_value:.4e}")

        if p_value < 0.05:
            print("Status: MISCALIBRATED (reject null)")
        else:
            print("Status: CALIBRATED (fail to reject null)")
        print("-" * 40)
    
    return ks_results, p_values

In [14]:
def multidim_coverage95(theta_est_post, theta_test_numpy):
    print("\nJoint 3D Coverage Metric:")
    print("-" * 40)

    num_test_samples = theta_test_numpy.shape[0]
    joint_covered_count = 0

    for i in range(num_test_samples):
        samples = theta_est_post[i]  # Shape: (1000, 3)
        truth = theta_test_numpy[i]  # Shape: (3,)

        mean_vec = np.mean(samples, axis=0)
        cov_matrix = np.cov(samples, rowvar=False)
        
        try:
            inv_cov = np.linalg.inv(cov_matrix)
        except np.linalg.LinAlgError:
            print(f"Warning: Singular covariance matrix at index {i}, skipping.")
            continue

        diff_samples = samples - mean_vec
        dist_samples = np.sum(np.dot(diff_samples, inv_cov) * diff_samples, axis=1)

        diff_truth = truth - mean_vec
        dist_truth = np.dot(np.dot(diff_truth, inv_cov), diff_truth)

        threshold_95 = np.percentile(dist_samples, 95)

        if dist_truth <= threshold_95:
            joint_covered_count += 1

    joint_coverage_95 = joint_covered_count / num_test_samples

    print(f"Target Joint Coverage: 95.0%")
    print(f"Actual Joint Coverage: {joint_coverage_95 * 100:.1f}%")
    print("-" * 40)

    return joint_coverage_95

In [15]:
def mean_error(theta_est_post, theta_test_numpy):
    post_mean_error = np.full((theta_est_post.shape[0]), np.nan)
    for i in range(theta_est_post.shape[0]):
        post_mean = np.mean(theta_est_post[i, :, :], axis=0)
        error = post_mean - theta_test_numpy[i, :]
        post_mean_error[i] = np.linalg.norm(error)
    return post_mean_error

## NLE

In [16]:
prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_theta = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_L = DiscreteUniform(low=torch.tensor([100.0]), high=torch.tensor([10000.0]))

prior = MultipleIndependent(
    dists=[prior_rho, prior_theta, prior_L],
    validate_args=False,
    device=torch_device
)

In [17]:
nle_sbc_D = np.full((3, len(budgets)), np.nan)
nle_sbc_p_values = np.full((3, len(budgets)), np.nan)
nle_coverage_results = np.full((2, len(budgets)), np.nan)
nle_mean_error_results = np.full((theta_test.shape[0], len(budgets), 2), np.nan)

nle_sbc_D.shape, nle_sbc_p_values.shape, nle_coverage_results.shape, nle_mean_error_results.shape

((3, 7), (3, 7), (2, 7), (998, 7, 2))

In [18]:
seed = 100
num_posterior_samples=1000
learning_rate = 0.0005

In [19]:
for i in range(len(budgets)):
    torch.manual_seed(seed)
    np.random.seed(seed)
    print("-" * 50)
    n_sim = budgets[i]
    x_train = x[:n_sim]
    theta_train = theta[:n_sim]

    print(f"\nTraining NLE with {n_sim} simulations")
    
    inference_benchmark = NLE_A(prior=prior, density_estimator="nsf", device=torch_device)
    density_estimator_benchmark = inference_benchmark.append_simulations(theta_train, x_train).train(
        max_num_epochs=500, learning_rate=learning_rate
    )
    posterior_benchmark = inference_benchmark.build_posterior(density_estimator_benchmark)

    print(f"Sampling posterior for NLE, n={n_sim}")

    theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
    for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
        theta_post = posterior_benchmark.sample((num_posterior_samples,), x=x_test[j, :],
                                                num_chains=1,
                                                show_progress_bars=True)
        theta_est_post[j, :, :] = theta_post.detach().numpy()
    
    print(f"Calculating metrics for NLE, n={n_sim}")

    sbc_results = []
    parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
    ranks, dap_samples = run_sbc(
        theta_test,
        x_test,
        posterior_benchmark,
        num_posterior_samples=num_posterior_samples,
        use_batched_sampling=False
    )

    ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)
    coverage_95_with_L = multidim_coverage95(theta_est_post, theta_test_numpy)
    coverage_95_without_L = multidim_coverage95(theta_est_post[:, :, :2], theta_test_numpy[:, :2])
    mean_error_with_L = mean_error(theta_est_post, theta_test_numpy)
    mean_error_without_L = mean_error(theta_est_post[:, :, :2], theta_test_numpy[:, :2])

    nle_sbc_D[:, i] = ks_results
    nle_sbc_p_values[:, i] = p_values
    nle_coverage_results[0, i] = coverage_95_with_L
    nle_coverage_results[1, i] = coverage_95_without_L
    nle_mean_error_results[:, i, 0] = mean_error_with_L
    nle_mean_error_results[:, i, 1] = mean_error_without_L

    print("-" * 50)

--------------------------------------------------

Training NLE with 200 simulations
 Neural network successfully converged after 136 epochs.Sampling posterior for NLE, n=200


Sampling posterior:   0%|          | 0/998 [00:27<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
np.savetxt("../data/benchmark/nle_sbc_D.csv", nle_sbc_D, delimiter=",")
np.savetxt("../data/benchmark/nle_sbc_p_values.csv", nle_sbc_p_values, delimiter=",")
np.savetxt("../data/benchmark/nle_coverage_results.csv", nle_coverage_results, delimiter=",")
np.save("../data/benchmark/nle_mean_error_results.npy", nle_mean_error_results)